# Exploratory Data Analysis — MIMIC-III Sepsis-3 Temporal Dataset

**Project:** Early Sepsis Prediction Using Temporal Deep Learning on ICU Data

This notebook provides the EDA needed for **Iteration 2: cohort construction, preprocessing, and data-quality evaluation**. It uses the processed data already stored in this repository.

The main questions are: cohort composition, Sepsis-3 onset timing, 4/8/12-hour eligibility, class imbalance after temporal filtering, feature coverage, missingness, and temporal clinical trajectories.

> The notebook deliberately separates **data-engineering success** from **scientific readiness**. The current 4h/8h tensors can be analysed, but the very small positive groups must be disclosed before model training is treated as a final experiment.

## 1. Setup and data loading

The path logic works when the notebook is run from the repository root or from the `notebooks/` directory.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

def locate_data_dir():
    candidates = [Path.cwd()/ 'data', Path.cwd().parent / 'data']
    for p in candidates:
        if (p / 'dataset_manifest.json').exists():
            return p.resolve()
    raise FileNotFoundError('Run this notebook from the cloned sepsis_test repository.')

DATA_DIR = locate_data_dir()
INTERIM = DATA_DIR / 'interim'
PROCESSED = DATA_DIR / 'processed'

manifest = json.loads((DATA_DIR / 'dataset_manifest.json').read_text())
onset_audit = json.loads((DATA_DIR / 'onset_eligibility_audit.json').read_text())
cohort = pd.read_csv(INTERIM / 'mimic_iii_sepsis3_cohort.csv', low_memory=False)
for c in ['admittime','dischtime','intime','outtime','sepsis_onset_time']:
    if c in cohort.columns:
        cohort[c] = pd.to_datetime(cohort[c], errors='coerce')

coverage = {}
tensors = {}
for h in [4,8]:
    coverage[h] = pd.read_csv(PROCESSED / f'feature_coverage_before_onset_{h}h.csv')
    z = np.load(PROCESSED / f'timeseries_before_onset_{h}h.npz', allow_pickle=False)
    tensors[h] = {k:z[k] for k in z.files}
    z.close()

print('Data directory:', DATA_DIR)
print('Cohort shape:', cohort.shape)
print('Available horizons:', sorted(tensors))

## 2. Cohort integrity and composition

The project retains the first ICU stay per patient to reduce patient-level leakage. The initial cohort should therefore contain unique patient and ICU identifiers.

In [ ]:
integrity = pd.DataFrame({
 'Measure':['Rows','Unique patients','Unique ICU stays','Duplicate patients','Duplicate ICU stays','Sepsis-3','Controls','Sepsis prevalence','Age <18','Positive missing onset'],
 'Value':[len(cohort),cohort.subject_id.nunique(),cohort.icustay_id.nunique(),cohort.subject_id.duplicated().sum(),cohort.icustay_id.duplicated().sum(),int((cohort.sepsis_label==1).sum()),int((cohort.sepsis_label==0).sum()),cohort.sepsis_label.mean(),int((cohort.age<18).sum()),int(((cohort.sepsis_label==1)&cohort.sepsis_onset_time.isna()).sum())]
})
display(integrity)

counts = cohort.sepsis_label.value_counts().sort_index()
plt.figure(figsize=(6,4))
bars=plt.bar(['Non-sepsis','Sepsis-3'],[counts.get(0,0),counts.get(1,0)])
plt.ylabel('Patients'); plt.title('Initial Sepsis-3 cohort class distribution')
for b in bars: plt.text(b.get_x()+b.get_width()/2,b.get_height(),f'{int(b.get_height()):,}',ha='center',va='bottom')
plt.tight_layout(); plt.show()

desc=[]
for label,g in cohort.groupby('sepsis_label'):
    desc.append({'Group':'Sepsis-3' if label else 'Non-sepsis','N':len(g),'Age median':g.age.median(),'Age Q1':g.age.quantile(.25),'Age Q3':g.age.quantile(.75),'Female %':100*g.gender.astype(str).str.upper().eq('F').mean(),'ICU LOS median days':g.icu_los_days.median(),'Hospital mortality %':100*pd.to_numeric(g.hospital_expire_flag,errors='coerce').mean()})
display(pd.DataFrame(desc).round(2))

plt.figure(figsize=(8,4))
for label,name in [(0,'Non-sepsis'),(1,'Sepsis-3')]:
    plt.hist(cohort.loc[cohort.sepsis_label==label,'age'].dropna(),bins=30,density=True,alpha=.5,label=name)
plt.xlabel('Age'); plt.ylabel('Density'); plt.title('Age distribution by outcome'); plt.legend(); plt.tight_layout(); plt.show()

los_cap=cohort.icu_los_days.quantile(.99)
a=cohort.loc[(cohort.sepsis_label==0)&(cohort.icu_los_days<=los_cap),'icu_los_days'].dropna()
b=cohort.loc[(cohort.sepsis_label==1)&(cohort.icu_los_days<=los_cap),'icu_los_days'].dropna()
plt.figure(figsize=(6,4)); plt.boxplot([a,b],tick_labels=['Non-sepsis','Sepsis-3'],showfliers=False); plt.ylabel('ICU LOS (days)'); plt.title('ICU length of stay by outcome'); plt.tight_layout(); plt.show()

## 3. Sepsis onset timing and temporal eligibility

For prediction horizon $h$ and a 12-hour input window:

$$t_{end}=t_{onset}-h$$

$$t_{start}=t_{end}-12h$$

A positive patient is eligible only if the complete observation window is inside the ICU stay. Therefore, a 4h, 8h, or 12h prediction generally requires onset at least 16h, 20h, or 24h after ICU admission.

In [ ]:
pos=cohort[cohort.sepsis_label==1].copy()
pos['onset_from_icu_hours']=(pos.sepsis_onset_time-pos.intime).dt.total_seconds()/3600
display(pos.onset_from_icu_hours.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).to_frame())

plt.figure(figsize=(9,4)); plt.hist(pos.onset_from_icu_hours.dropna(),bins=70)
plt.axvline(0,ls='--',label='ICU admission'); plt.axvline(16,ls=':',label='4h + 12h input'); plt.axvline(20,ls=':',label='8h + 12h input'); plt.axvline(24,ls=':',label='12h + 12h input')
plt.xlabel('Onset relative to ICU admission (hours)'); plt.ylabel('Patients'); plt.title('Sepsis-3 onset timing'); plt.legend(); plt.tight_layout(); plt.show()

def eligible_cases(h,input_hours=12):
    c=cohort[cohort.sepsis_label==1].copy(); onset=c.sepsis_onset_time
    end=onset-pd.to_timedelta(h,unit='h'); start=end-pd.to_timedelta(input_hours,unit='h')
    ok=onset.notna() & onset.between(c.intime,c.outtime,inclusive='both') & start.ge(c.intime) & end.le(c.outtime)
    return int(ok.sum())

elig=pd.DataFrame({'Horizon':['4h','8h','12h'],'Eligible Sepsis-3':[eligible_cases(4),eligible_cases(8),eligible_cases(12)]})
display(elig)
plt.figure(figsize=(6,4)); bars=plt.bar(elig.Horizon,elig['Eligible Sepsis-3']); plt.ylabel('Eligible positives'); plt.title('Positive eligibility after temporal-window constraints')
for b in bars: plt.text(b.get_x()+b.get_width()/2,b.get_height(),str(int(b.get_height())),ha='center',va='bottom')
plt.tight_layout(); plt.show()

initial=int((cohort.sepsis_label==1).sum())
attr=pd.DataFrame({'Stage':['Initial positives','4h eligible','8h eligible','12h eligible'],'Positive patients':[initial,eligible_cases(4),eligible_cases(8),eligible_cases(12)]})
attr['Retention %']=100*attr['Positive patients']/initial
display(attr.round(3))

### Critical interpretation

The initial cohort is approximately balanced, but the temporal eligibility rule removes almost all positive patients. This means the main class-imbalance problem is created **after** cohort construction by the interaction between onset timing, the 12-hour observation window, and the prediction gap. This should be explicitly discussed in the Iteration 2 evaluation.

## 4. Processed temporal tensors and post-filter class imbalance

The intended input structure is $X\in\mathbb{R}^{N\times12\times22}$: patients × hourly time steps × clinical features.

In [ ]:
rows=[]
for h,d in tensors.items():
    y=d['y'].astype(int); x=d['x']
    rows.append({'Horizon':f'{h}h','X shape':str(tuple(x.shape)),'Patients':len(y),'Sepsis-3':int(y.sum()),'Controls':int((y==0).sum()),'Positive prevalence %':100*y.mean(),'Unique patients':len(np.unique(d['subject_id'])),'Unique ICU stays':len(np.unique(d['icustay_id']))})
display(pd.DataFrame(rows).round(3))

hs=sorted(tensors); neg=[int((tensors[h]['y']==0).sum()) for h in hs]; sep=[int(tensors[h]['y'].sum()) for h in hs]; xx=np.arange(len(hs)); w=.35
plt.figure(figsize=(7,4)); plt.bar(xx-w/2,neg,w,label='Controls'); plt.bar(xx+w/2,sep,w,label='Sepsis-3'); plt.xticks(xx,[f'{h}h' for h in hs]); plt.ylabel('Samples'); plt.title('Class distribution after temporal filtering'); plt.legend(); plt.tight_layout(); plt.show()

## 5. Clinical feature coverage and missingness

Forward filling is performed only within each patient's observation window. The EDA compares raw coverage with post-forward-fill coverage. Remaining missing values must later be imputed using **training-only statistics** to avoid leakage.

In [ ]:
feature_names=tensors[4]['feature_names'].astype(str).tolist(); display(pd.DataFrame({'Feature':feature_names}))

for h in sorted(tensors):
    cov=coverage[h].sort_values('post_ffill_coverage')
    yy=np.arange(len(cov)); plt.figure(figsize=(9,7))
    plt.barh(yy-.18,cov.raw_coverage,height=.35,label='Raw observed'); plt.barh(yy+.18,cov.post_ffill_coverage,height=.35,label='After forward fill')
    plt.yticks(yy,cov.feature); plt.xlabel('Coverage fraction'); plt.xlim(0,1); plt.title(f'Feature coverage — {h}h'); plt.legend(); plt.tight_layout(); plt.show()

for h,d in tensors.items():
    x=d['x'].astype(float); y=d['y'].astype(int); miss=np.isnan(x).mean(axis=(1,2)); allmiss=np.isnan(x).all(axis=(1,2))
    print(f'{h}h: mean missing={miss.mean():.3f}, median missing={np.median(miss):.3f}, all-missing windows={allmiss.sum()}, positive all-missing={y[allmiss].sum() if allmiss.any() else 0}')
    plt.figure(figsize=(7,4)); plt.hist(miss,bins=30); plt.xlabel('Fraction missing'); plt.ylabel('Patients'); plt.title(f'Patient-level missingness — {h}h'); plt.tight_layout(); plt.show()
    non=miss[y==0]; sep=miss[y==1]; plt.figure(figsize=(6,4)); plt.boxplot([non,sep],tick_labels=['Non-sepsis','Sepsis-3'],showfliers=False); plt.ylabel('Fraction missing'); plt.title(f'Missingness by class — {h}h'); plt.tight_layout(); plt.show()

## 6. Temporal coverage heatmaps

These heatmaps show the fraction of patients with an observed value at each feature-hour position. They reveal both feature-level and time-level sparsity.

In [ ]:
for h,d in tensors.items():
    observed=np.isfinite(d['x'].astype(float)).mean(axis=0).T
    plt.figure(figsize=(11,8)); im=plt.imshow(observed,aspect='auto',vmin=0,vmax=1); plt.xticks(np.arange(12),np.arange(1,13)); plt.yticks(np.arange(len(feature_names)),feature_names); plt.xlabel('Observation hour'); plt.ylabel('Feature'); plt.title(f'Temporal feature coverage — {h}h'); plt.colorbar(im,label='Observed fraction'); plt.tight_layout(); plt.show()

## 7. Class-specific measurement availability

Measurement intensity can itself carry information: sicker patients may receive more frequent tests. The following analysis checks the largest differences in observed-feature coverage between Sepsis-3 and controls. With only 29 positive cases at 4h and 14 at 8h, this is descriptive only.

In [ ]:
for h,d in tensors.items():
    x=d['x'].astype(float); y=d['y'].astype(int)
    control=np.isfinite(x[y==0]).mean(axis=(0,1)); sepsis=np.isfinite(x[y==1]).mean(axis=(0,1))
    tab=pd.DataFrame({'Non-sepsis':control,'Sepsis-3':sepsis},index=feature_names); tab['Difference']=tab['Sepsis-3']-tab['Non-sepsis']; top=tab.reindex(tab.Difference.abs().sort_values(ascending=False).head(10).index)
    print(f'\n{h}h horizon'); display(top.round(3))
    xx=np.arange(len(top)); w=.35; plt.figure(figsize=(10,4)); plt.bar(xx-w/2,top['Non-sepsis'],w,label='Non-sepsis'); plt.bar(xx+w/2,top['Sepsis-3'],w,label='Sepsis-3'); plt.xticks(xx,top.index,rotation=35,ha='right'); plt.ylabel('Observed fraction'); plt.title(f'Largest feature-availability differences — {h}h'); plt.legend(); plt.tight_layout(); plt.show()

## 8. Temporal trajectories of key variables

The following curves use the observed values already present in the processed tensors. No final mean imputation or scaling is applied during EDA. Because the positive groups are extremely small, the curves should be interpreted as exploratory rather than stable population estimates.

In [ ]:
key=['heart_rate','resp_rate','spo2','temperature_c','sbp','glucose','lactate','wbc']
for h,d in tensors.items():
    names=d['feature_names'].astype(str).tolist(); x=d['x'].astype(float); y=d['y'].astype(int); hours=np.arange(1,13)
    print(f'\n--- {h}h prediction horizon ---')
    for f in key:
        if f not in names: continue
        j=names.index(f); plt.figure(figsize=(7,4))
        for label,name in [(0,'Non-sepsis'),(1,'Sepsis-3')]:
            s=x[y==label,:,j]; mean=np.nanmean(s,axis=0); plt.plot(hours,mean,marker='o',label=f'{name} (n={len(s)})')
        plt.xlabel('Observation hour'); plt.ylabel(f); plt.title(f.replace('_',' ').title()+f' — {h}h'); plt.xticks(hours); plt.legend(); plt.tight_layout(); plt.show()

## 9. EDA conclusion for Iteration 2

The processing pipeline successfully produced structured **12 × 22 temporal tensors** for the 4-hour and 8-hour settings. The original cohort contains **11,302 unique patients** and is approximately balanced.

The dominant limitation appears after temporal-window filtering. Only a very small number of positive patients have enough pre-onset history for the planned 12-hour observation window, and the current build has no valid 12-hour-horizon positive cohort. Therefore, the later modelling notebook should use accuracy only together with precision, recall, F1, AUROC, AUPR, confusion matrices, and uncertainty analysis.

Feature coverage is also heterogeneous. Frequently monitored vital signs are highly available, while several laboratory measurements remain sparse even after within-window forward filling. Remaining imputation and standardisation should be fitted using the training partition only.

### Best evidence to include in the report

1. Initial cohort class distribution
2. Cohort descriptive-statistics table
3. Sepsis-onset timing histogram
4. Positive eligibility / attrition table or figure
5. 4h/8h class distribution after temporal filtering
6. Feature-coverage plot
7. Temporal feature-coverage heatmap
8. Missingness summary

These plots directly support the **Development of Iteration 2** and **Review and Evaluation of Iteration 2** sections of the dissertation.